In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.ticker import FormatStrFormatter
import seaborn as sns
import numpy as np
import gzip
import upsetplot
from collections import defaultdict
import itertools

In [ ]:
poly_homopolymer_regions = [302,303,304,305,306,307,308,309,310,311,16179,16180,16181,16182,16183,3106,3107]
excluded_samples = ['ST002-1D_LUNG-pacbio-uwsc-group1']

In [ ]:
## read in mitoscope qc files
df_pb = pd.read_csv("../benchmark/pacbio/output/qc_summary.tsv", sep='\t')
df_pb[['Donor', 'Tissue', 'Seq_Tech', 'Center', 'Group']] = df_pb['Sample'].str.split('-', expand=True)
df_pb['Age'] = np.where(df_pb['Donor'].isin(['ST001', 'ST003']), 'Young', 'Old')

df_ont = pd.read_csv("../benchmark/ont/output/qc_summary.tsv", sep='\t')
df_ont[['Donor', 'Tissue', 'Seq_Tech', 'Center']] = df_ont['Sample'].str.split('-', expand=True)
df_ont['Age'] = np.where(df_ont['Donor'].isin(['ST001', 'ST003']), 'Young', 'Old')

df = pd.concat([df_pb, df_ont]).reset_index()
df['Donor_Tissue'] = df['Donor'] + "-" + df['Tissue']
df = df.sort_values(['Tissue', 'Donor', 'Center'])

df = df[~(df['Sample'].isin(excluded_samples))]
df['Tissue'] = df['Tissue'].str.split('_', expand=True)[1].str.capitalize()
df['Seq_Tech'] = np.where(df['Seq_Tech'] == 'pacbio', 'PacBio', 'ONT')

df

In [ ]:
### pull in matched-short read data to compare CN results against long-reads
sr_manifest=pd.read_csv("/net/nwgc/vol1/home/czaka/analysis/mutect2/smaht/illumina/benchmark/samples_full.csv", header=None, names=['sample', 'path'])

for sample in sr_manifest['sample']:
    mito_coverage_file=f'/net/nwgc/vol1/home/czaka/analysis/mutect2/smaht/illumina/benchmark/mosdepth/{sample}.mosdepth.summary.txt'
    mito_cov_df = pd.read_csv(mito_coverage_file, sep='\t')
    mito_cov = mito_cov_df[mito_cov_df['chrom'] == 'chrM']['mean'].iloc[0]
    sr_manifest.loc[sr_manifest['sample'] == sample, 'mt_cov'] = mito_cov
    
    nuc_coverage_file=f'/net/nwgc/vol1/home/czaka/analysis/mutect2/smaht/illumina/benchmark/mosdepth/nuclear/{sample}.mosdepth.summary.txt'
    nuc_cov_df = pd.read_csv(nuc_coverage_file, sep='\t')
    nuc_cov = nuc_cov_df[nuc_cov_df['chrom'] == 'total']['mean'].iloc[0]
    sr_manifest.loc[sr_manifest['sample'] == sample, 'nuc_cov'] = nuc_cov

    cn = (mito_cov / nuc_cov) * 2
    sr_manifest.loc[sr_manifest['sample'] == sample, 'mtDNA_CN'] = cn

sr_manifest[['Donor', 'Tissue', 'Seq_Tech', 'Center']] = sr_manifest['sample'].str.split('-', expand=True)
sr_manifest['Tissue'] = sr_manifest['Tissue'].str.split('_', expand=True)[1].str.capitalize()
sr_manifest['Seq_Tech'] = sr_manifest['Seq_Tech'].str.capitalize()
sr_manifest['Donor_Tissue'] = sr_manifest['Donor'] + "-" + sr_manifest['Tissue']

sr_manifest

In [ ]:
sr_manifest['Donor_Tissue'].value_counts()

In [ ]:
## merge SR and LR copy number results

sr_lr_cn_df = pd.concat([sr_manifest[['Donor', 'Tissue', 'Seq_Tech', 'Center', 'mtDNA_CN']],df[['Donor', 'Tissue', 'Seq_Tech', 'Center', 'mtDNA_CN']]])
sr_lr_cn_df['Donor_Tissue'] = sr_lr_cn_df['Donor'] + '-' + sr_lr_cn_df['Tissue']
sr_lr_cn_df

In [ ]:
## ILLUMINA VARIANT CALLS
def read_vcf(input_file):
    with gzip.open(input_file, 'rt') as fr:
        for line in fr:
            if line.startswith('#CHROM'):
                header = line.strip().lstrip('#').split('\t')
                break

    df = pd.read_csv(input_file, comment='#', sep='\t', compression='gzip', names=header)
    return df

snv_vcf_ill = read_vcf("../../../mutect2/smaht/illumina/benchmark_byTissue/output/merged.mutect2.vcf.gz")
snv_vcf_ill['ID'] = 'MT-' + snv_vcf_ill[['POS', 'REF', 'ALT']].astype(str).agg('-'.join, axis=1)

filter_col = [col for col in snv_vcf_ill if col.startswith('ST00')]
snv_df_ill = pd.melt(
    snv_vcf_ill[["POS", "ID", "INFO"] + filter_col],
    id_vars=['POS', 'ID', "INFO"],
    var_name='Sample',
    value_name='Value'
)

snv_df_ill[['Donor', 'Tissue_Code', 'Seq_Tech',]] = snv_df_ill['Sample'].str.split('-', expand=True)

tissue_dict = {'1A':'Liver', '1D':'Lung', '1G':'Colon', '1Q':'Brain'}
snv_df_ill['Tissue'] = snv_df_ill['Tissue_Code'].map(tissue_dict)

snv_df_ill['Donor_Tissue'] = snv_df_ill['Donor'] + "_" + snv_df_ill['Tissue']


## more formatting + add sample specific and total replicate support
snv_df_ill['AF'] = snv_df_ill['Value'].str.split(':', expand=True)[2]
snv_df_ill = snv_df_ill.drop(columns=['Value', 'INFO'])
snv_df_ill = snv_df_ill[(snv_df_ill['AF'] != '.')]


snv_df_ill[['MT', 'POS', 'REF', 'ALT']] = snv_df_ill['ID'].str.split('-', expand=True)
snv_df_ill['indel'] = snv_df_ill.apply(lambda row: 'indel' if (len(row['REF']) > 1 or len(row['ALT']) > 1) else 'snv', axis=1)
snv_df_ill['AF'] = snv_df_ill['AF'].astype(float)
snv_df_ill['heteroplasmy_category'] = np.where(snv_df_ill['AF'] > 0.9, 'homo', 'hetero')

snv_df_ill = snv_df_ill[~(snv_df_ill['POS'].isin(poly_homopolymer_regions))]
snv_df_ill = snv_df_ill[snv_df_ill['indel'] == 'snv']

snv_df_ill


In [ ]:
# ## plot read count, coverage, length stats
# sns.set_theme(style="ticks", context="talk", font_scale=0.7)

# for category in ["Mito_Read_Count", "Mito_Coverage", "Nuclear_Coverage", "Mean_Read_Length"]:

#     fig, ax = plt.subplots(layout='constrained', figsize=(7, 5))

#     sns.boxplot(
#         data=df,
#         x="Donor_Tissue",
#         y=category,
#         hue="Seq_Tech",
#         showfliers=True,
#         palette="Set2",
#         widths=0.3,
#         legend=False,
#         ax=ax
#     )

#     # Get original tick labels
#     ticks = ax.get_xticklabels()
#     labels = [t.get_text() for t in ticks]
#     donors  = [l.split("-")[0] for l in labels]  
#     tissues = ['\n\n' + l.split("-")[1] for l in labels]

#     ax.set_xticks(range(len(labels)))
#     ax.set_xticklabels(donors, rotation=0)
#     ax.set_ylabel(category)
#     ax.set_xlabel("")

#     sec = ax.secondary_xaxis(location=0)
#     sec.set_xticks([0,1.5,3,4.5], labels=['\n\nLiver', '\n\nLung', '\n\nColon', '\n\nBrain'])
#     sec.tick_params('x', length=0)

#     for tick in sec.get_xticklabels():
#         tick.set_fontweight("bold")
#         #tick.set_fontsize(14)

#     midpoints = [0.5,2.5,3.5]
#     for x in midpoints:
#         ax.axvline(x=x,color="gray",linestyle="--",linewidth=1,alpha=1,zorder=0)

#     #plt.yscale('log')
#     print(tissues)
#     #plt.savefig(f"plots/fig3-benchmark_tissue_QC_{category}.pdf", dpi=300)
#     plt.show()

In [ ]:
## plot mtDNA CN
sns.set_theme(style="ticks", context="talk", font_scale=0.7)

for category in ["mtDNA_CN"]:

    fig, ax = plt.subplots(layout='constrained', figsize=(6, 5))

    sns.boxplot(
        data=sr_lr_cn_df,
        x="Donor_Tissue",
        y=category,
        hue="Seq_Tech",
        showfliers=True,
        palette="Set2",
        widths=0.2,
        legend=True,
        ax=ax
    )

    # Get original tick labels
    ticks = ax.get_xticklabels()
    labels = [t.get_text() for t in ticks]
    donors  = [l.split("-")[0] for l in labels]  
    tissues  = [l.split("-")[1] for l in labels]  

    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(donors, rotation=0)
    ax.set_ylabel("mtDNA Copy Number")
    ax.set_xlabel("")

    sec = ax.secondary_xaxis(location=0)
    sec.set_xticks([0,1.5,3,4.5], labels=['\n\nLiver', '\n\nLung', '\n\nColon', '\n\nBrain'])
    sec.tick_params('x', length=0)

    for tick in sec.get_xticklabels():
        tick.set_fontweight("bold")
        #tick.set_fontsize(14)

    midpoints = [0.5,2.5,3.5]
    for x in midpoints:
        ax.axvline(x=x,color="gray",linestyle="--",linewidth=1,alpha=1,zorder=0)

    plt.yscale('log')
    print(tissues)
    print(donors)
    plt.savefig(f"plots_revisions/fig3-benchmark_tissue_QC_{category}.pdf", dpi=300)
    plt.show()

In [ ]:
## plot mt coverage vs nuc coverage 
sns.set_theme(style="ticks", context="talk", font_scale=0.75)

plt.figure(figsize=(6,5))
sns.scatterplot(
    data=df,
    x="Nuclear_Coverage",
    y="Mito_Coverage",
    hue="Tissue",     
    palette="muted",
    s=70,
    style="Seq_Tech",
    alpha=1)
plt.xlabel("Nuclear Genome Coverage")
plt.ylabel("Mitochondria Genome Coverage")
plt.yscale("log")
#plt.xscale("log")
plt.title("")
plt.legend(title="", loc='upper right', fontsize=11)
plt.tight_layout()
#plt.savefig("plots/suppl/benchmark_tissue_coverage.pdf", dpi=300)
plt.show()



In [ ]:
## generate combined df with variants called across all PB and ONT samples
def read_vcf(input_file):
    with gzip.open(input_file, 'rt') as fr:
        for line in fr:
            if line.startswith('#CHROM'):
                header = line.strip().lstrip('#').split('\t')
                break

    df = pd.read_csv(input_file, comment='#', sep='\t', compression='gzip', names=header)
    return df

snv_vcf_pb = read_vcf("../benchmark/pacbio_byTissue/output/merged.mt.baldur.annotated.vcf.gz")
snv_vcf_ont = read_vcf("../benchmark/ont_byTissue/output/merged.mt.baldur.annotated.vcf.gz")

snv_vcf_pb['ID'] = snv_vcf_pb[['CHROM', 'POS', 'REF', 'ALT']].astype(str).agg('-'.join, axis=1)
snv_vcf_ont['ID'] = snv_vcf_ont[['CHROM', 'POS', 'REF', 'ALT']].astype(str).agg('-'.join, axis=1)

filter_col = [col for col in snv_vcf_pb if col.startswith('ST00')]
snv_df_pb = pd.melt(
    snv_vcf_pb[["POS", "ID", "INFO"] + filter_col],
    id_vars=['POS', 'ID', "INFO"],
    var_name='Sample',
    value_name='Value'
)

filter_col = [col for col in snv_vcf_ont if col.startswith('ST00')]
snv_df_ont = pd.melt(
    snv_vcf_ont[["POS", "ID", "INFO"] + filter_col],
    id_vars=['POS', 'ID', "INFO"],
    var_name='Sample',
    value_name='Value'
)

snv_df = pd.concat([snv_df_pb, snv_df_ont])

snv_df[['Donor', 'Tissue_Code', 'Seq_Tech']] = snv_df['Sample'].str.split('-', expand=True)

#snv_df['Tissue'] = snv_df['Tissue'].str.split("_", expand=True)[1].str.capitalize()
tissue_dict = {'1A':'Liver', '1D':'Lung', '1G':'Colon', '1Q':'Brain'}
snv_df['Tissue'] = snv_df['Tissue_Code'].map(tissue_dict)

snv_df['Donor_Tissue'] = snv_df['Donor'] + "_" + snv_df['Tissue']

snv_df['VEP'] = snv_df['INFO'].str.extract(r'CSQ=(.+);')
snv_df[['Consequence', 'Impact', 'Symbol', 'Biotype']] = snv_df['VEP'].str.split('|', expand=True)[[1,2,3,7]]

snv_df['MITOMAP'] = snv_df['INFO'].str.extract(r'MITOMAP=(.+);')
snv_df[['Gene.Name', 'Gene.Type', 'Amino.Acid.Change', 'GB.Freq.FL', 'GB.Freq.CR', 'GB.Seqs.FL', 'GB.Seqs.CR', 'Disease', 'Status', 'Additional.Annotations', 'MitoTIP']] = snv_df['MITOMAP'].str.split('|', expand=True)[[0,1,2,3,4,5,6,9,10,12,13]]

snv_df = snv_df[~(snv_df['Sample'].isin(excluded_samples))]

snv_df


In [ ]:
## more formatting + add sample specific and total replicate support
snv_df[['GT', 'ADF', 'ADR','HPL', 'FQSE', 'AQ', 'AFLT', 'QAVG', 'FSB', 'QBS']] = snv_df['Value'].str.split(':', expand=True)
snv_df = snv_df.drop(columns=['Value', 'INFO'])
snv_df = snv_df[(snv_df['GT'] != '.') & (snv_df['GT'] != './.') & (snv_df['GT'] != '././.') & (snv_df['GT'] != './././.') & (snv_df['GT'] != '././././.')]
snv_df['AF'] = snv_df['HPL']

### note - this doesn't seem to be necessary for baldur, bcftools norm takes care as expected ###
def extract_element(row):
    if ',' in row['AF']:
        parts = row['AF'].split(',')
        if row['GT'] == '0/1':
            return parts[1]
        elif row['GT'] == '1/0':
            return parts[0]
    return row['AF']

snv_df['AF'] = snv_df.apply(extract_element, axis=1)
snv_df['AF'] = snv_df['AF'].astype(float)

snv_df[['MT', 'POS', 'REF', 'ALT']] = snv_df['ID'].str.split('-', expand=True)
snv_df['POS'] = snv_df['POS'].astype(int)
snv_df['indel'] = snv_df.apply(lambda row: 'indel' if (len(row['REF']) > 1 or len(row['ALT']) > 1) else 'snv', axis=1)
snv_df['heteroplasmy_category'] = np.where(snv_df['AF'] > 0.95, 'homo', 'hetero')

snv_df['reps_sample_specific'] = (
    snv_df
    .groupby(['ID','Donor','Tissue'])['ID']
    .transform('count')
)

snv_df['reps_donor_specific'] = (
    snv_df
    .groupby(['ID','Donor'])['ID']
    .transform('count')
)

snv_df['reps_total'] = (
    snv_df
    .groupby(['ID'])['ID']
    .transform('count')
)

snv_df['AF'] = (
    snv_df
    .groupby(['ID','Donor','Tissue'])['AF']
    .transform('mean')
)

def custom_join(series):
    return ','.join(series.astype(str))

snv_df['reps_sample_specific_names'] = (
    snv_df
    .groupby(['ID','Donor','Tissue'])['Seq_Tech']
    .transform(custom_join)
)

snv_df['reps_donor_specific_names'] = (
    snv_df
    .groupby(['ID','Donor'])['Sample']
    .transform(custom_join)
)

snv_df['reps_total_names'] = (
    snv_df
    .groupby(['ID'])['Sample']
    .transform(custom_join)
)

#snv_df = snv_df[~(snv_df['POS'].isin(poly_homopolymer_regions))]
snv_df = snv_df[snv_df['AF'] >= 0.0025]

snv_df

In [ ]:
## format annotation columns
def custom_join(series):
    return ','.join(series.astype(str))

collapsed_snv_df = snv_df.drop_duplicates(subset=['ID', 'Donor', 'Tissue']).drop(
    columns=['Sample', 'Seq_Tech', 'VEP', 'MITOMAP', 'POS', 
             'GT', 'ADF', 'ADR', 'HPL', 'FQSE', 'AQ', 'AFLT', 'QAVG', 'FSB', 'QBS', 'MT', 'REF', 'ALT'])

collapsed_snv_df['heteroplasmy_category'] = np.where(collapsed_snv_df['AF'] > 0.95, 'homo', 'hetero')

collapsed_snv_df['Status'] = np.where(collapsed_snv_df['Status'] == 'Reported%3B lineage L & M marker, also hg IJK', 'Reported', collapsed_snv_df['Status'])
collapsed_snv_df['Status'] = np.where(collapsed_snv_df['Status'] == '', 'N/A', collapsed_snv_df['Status'])

collapsed_snv_df['Gene.Type'] = collapsed_snv_df['Gene.Type'].fillna('')
collapsed_snv_df['Gene.Name'] = collapsed_snv_df['Gene.Name'].fillna('')

collapsed_snv_df['Gene.Type'] = np.where(collapsed_snv_df['Gene.Type'] == '', collapsed_snv_df['Biotype'], collapsed_snv_df['Gene.Type'])
collapsed_snv_df['Gene.Type'] = np.where(collapsed_snv_df['Gene.Type'] == 'protein_coding', 'protein coding', collapsed_snv_df['Gene.Type'])

# for the one variant that is in a noncoding region 7515-7517 isn't being labeled by mitomap
collapsed_snv_df['Gene.Type'] = np.where(collapsed_snv_df['Gene.Type'] == '', 'noncoding', collapsed_snv_df['Gene.Type'])

collapsed_snv_df['Consequence'] = (
    collapsed_snv_df['Consequence']
    .str.split('_')
    .apply(lambda x: '_'.join(x[:-1]))
)

indel_df = collapsed_snv_df[collapsed_snv_df['indel'] == 'indel']


collapsed_snv_df[['MT', 'POS', 'REF', 'ALT']] = collapsed_snv_df['ID'].str.split('-', expand=True)
collapsed_snv_df['POS'] = collapsed_snv_df['POS'].astype(int)
collapsed_snv_df['homopolymer'] = np.where(collapsed_snv_df['POS'].isin(poly_homopolymer_regions), 'yes', 'no')

homopoly_calls = collapsed_snv_df[collapsed_snv_df['homopolymer'] == 'yes']

collapsed_snv_df = collapsed_snv_df[collapsed_snv_df['homopolymer'] == 'no']
collapsed_snv_df = collapsed_snv_df[collapsed_snv_df['indel'] == 'snv']

collapsed_snv_df


In [ ]:
homopoly_calls['reps_sample_specific_names'].value_counts()

In [ ]:
## 245 // 192 shared, 32, 21
# Of these 25 were in homopolymer regions (15 SNVs, 10 indels all heteroplasmic; only 6 shared between PB and ONT (10 uniq PB, 9 ONT)) -> high discordance,

### 220 left (219 SNVs, 1 indel), SNVs -> 166 homo, 53 hetero
## 20/53 shared, 21 unique to PB, 12 to ONT (41% shared)


collapsed_snv_df[['reps_sample_specific_names', 'heteroplasmy_category', 'indel']].value_counts()

In [ ]:
## read in duplex pileup results

def read_vcf(input_file):
    with gzip.open(input_file, 'rt') as fr:
        for line in fr:
            if line.startswith('#CHROM'):
                header = line.strip().lstrip('#').split('\t')
                break

    df = pd.read_csv(input_file, comment='#', sep='\t', compression='gzip', names=header)
    return df

duplex_sample_manifest = pd.read_csv('/net/nwgc/vol1/home/czaka/analysis/codec/benchmark/sample_vcf_list.txt', sep=',', names=['sample', 'path'])

comb_duplex_df = pd.DataFrame()
for index,row in duplex_sample_manifest.iterrows():
    df1 = read_vcf(row['path'])
    comb_duplex_df = pd.concat([comb_duplex_df, df1])

comb_duplex_df['ID'] = 'MT-' + comb_duplex_df[['POS', 'REF', 'ALT']].astype(str).agg('-'.join, axis=1)
filter_col = [col for col in comb_duplex_df if col.startswith('ST00')]

comb_duplex_df = pd.melt(
    comb_duplex_df[["POS", "ID"] + filter_col],
    id_vars=['POS', 'ID'],
    var_name='Sample',
    value_name='Value'
)

comb_duplex_df = comb_duplex_df[~comb_duplex_df['Value'].isna()]
comb_duplex_df[['Donor', 'Tissue', 'Seq_Tech', 'Center']] = comb_duplex_df['Sample'].str.split('-', expand=True)
comb_duplex_df['Tissue'] = comb_duplex_df['Tissue'].str.split('_', expand=True)[1].str.capitalize()
comb_duplex_df['Donor_Tissue'] = comb_duplex_df['Donor'] + "_" + comb_duplex_df['Tissue']
comb_duplex_df[['GT', 'ADF', 'ADR', 'DPF', 'DPR', 'AF']] = comb_duplex_df['Value'].str.split(':', expand=True)
comb_duplex_df[['MT', 'POS', 'REF', 'ALT']] = comb_duplex_df['ID'].str.split('-', expand=True)
comb_duplex_df['POS'] = comb_duplex_df['POS'].astype(int)
comb_duplex_df['indel'] = comb_duplex_df.apply(lambda row: 'indel' if (len(row['REF']) > 1 or len(row['ALT']) > 1) else 'snv', axis=1)
comb_duplex_df['heteroplasmy_category'] = np.where(comb_duplex_df['AF'].astype(float) > 0.95, 'homo', 'hetero')

comb_duplex_df = comb_duplex_df[~(comb_duplex_df['POS'].isin(poly_homopolymer_regions))]
comb_duplex_df = comb_duplex_df[comb_duplex_df['indel'] == 'snv']

comb_duplex_df

In [ ]:
comb_duplex_df['indel'].value_counts()

In [ ]:
i = pd.merge(snv_df_ill[snv_df_ill['indel'] == 'indel'], comb_duplex_df[comb_duplex_df['indel'] == 'indel'], on=['ID', 'Donor_Tissue'], how='outer', indicator=True)

i[i['_merge'] == 'both']

In [ ]:
merged_w_dup = collapsed_snv_df.merge(
    comb_duplex_df,
    on=['Donor_Tissue', 'ID', 'Donor', 'Tissue', 'indel', 'heteroplasmy_category'],
    how='outer',
    indicator=True
)
merged_w_dup = merged_w_dup.rename(columns={"_merge": "duplex_merge", "AF_x": "AF_mito", "AF_y": "AF_dup"})
merged_w_dup['AF_dup'] = merged_w_dup['AF_dup'].astype(float)

merged_w_dup['overall_status'] = np.select(
    [
        merged_w_dup['duplex_merge'].eq('both'),
        merged_w_dup['duplex_merge'].eq('right_only'),
        merged_w_dup['duplex_merge'].eq('left_only')
    ],
    [
        merged_w_dup['reps_sample_specific_names'].astype(str) + ',duplex',
        'duplex',
        merged_w_dup['reps_sample_specific_names'].astype(str)
    ],
    default=merged_w_dup['reps_sample_specific_names'].astype(str)
)

merged_w_dup['overall_status'].value_counts()



In [ ]:
duplex_only = merged_w_dup[merged_w_dup['overall_status'] == 'duplex']

## plot mito coverage for hapmap illumina, pb, and ont 
sns.set_theme(style="ticks", context="talk", font_scale=0.75)
plt.figure(figsize=(6, 4))

sns.histplot(
    duplex_only,
    x='AF_dup',
    bins=50
)

plt.xlabel('Heteroplasmy Fraction')
plt.ylabel('Frequency')

plt.tight_layout()
plt.savefig("plots_revisions/suppl/duplex_vafs.pdf", dpi=300)
plt.show()

In [ ]:
merged = merged_w_dup.merge(
    snv_df_ill[['ID', 'Donor_Tissue', 'AF', 'indel']],
    on=['Donor_Tissue', 'ID', 'indel'],
    how='outer',
    indicator=True
)
merged = merged.rename(columns={"_merge": "sr_merge", "AF": "AF_sr"})

merged['overall_status'] = np.select(
    [
        merged['sr_merge'].eq('both'),
        merged['sr_merge'].eq('right_only'),
        merged['sr_merge'].eq('left_only')
    ],
    [
        merged['overall_status'].astype(str) + ',illumina',
        'illumina',
        merged['overall_status'].astype(str)
    ],
    default=merged['overall_status'].astype(str)
)

merged['heteroplasmy_category'] = np.select(
    [
        (merged['heteroplasmy_category'].isna()) & (merged['AF_sr'] > 0.95),
        (merged['heteroplasmy_category'].isna()) & (merged['AF_sr'] <= 0.95)
    ],
    [
        'homo',
        'hetero'
    ],
    default=merged['heteroplasmy_category']
)

merged


In [ ]:
merged[['overall_status', 'heteroplasmy_category']].value_counts()

In [ ]:
merged[merged['heteroplasmy_category'] == 'homo'][merged['overall_status'] == 'pacbio,ont,duplex']

In [ ]:
## upset plot // 1 for homo, 1 for hetero

upset_df = merged[['ID', 'Donor_Tissue', 'overall_status', 'heteroplasmy_category']]

sns.set_theme(style="ticks", context="talk", font_scale=0.65)
tt = upsetplot.from_memberships(upset_df[upset_df['heteroplasmy_category'] == 'homo'].overall_status.str.split(","), data=upset_df[upset_df['heteroplasmy_category'] == 'homo'])
fig = plt.figure(figsize=(7,6))
ttt = upsetplot.UpSet(tt, element_size=None, sort_categories_by="input", sort_by="-degree", show_percentages=True, show_counts=True,totals_plot_elements=0)

ttt.style_subsets(facecolor='blue', min_degree=1, max_degree=1)
ttt.style_subsets(facecolor='green', min_degree=2, max_degree=2)
ttt.style_subsets(facecolor='orange', min_degree=3, max_degree=3)
ttt.style_subsets(facecolor='purple', min_degree=4, max_degree=4)
ttt.style_subsets(facecolor='red', min_degree=5, max_degree=5)

ttt.plot(fig)
plt.suptitle("Homoplasmic", size=16)
fig.savefig("plots_revisions/suppl/upset_plot_homo.pdf", dpi=300)
plt.show()

sns.set_theme(style="ticks", context="talk", font_scale=0.65)
tt = upsetplot.from_memberships(upset_df[upset_df['heteroplasmy_category'] == 'hetero'].overall_status.str.split(","), data=upset_df[upset_df['heteroplasmy_category'] == 'hetero'])
fig = plt.figure(figsize=(12,6))
ttt = upsetplot.UpSet(tt, element_size=None, sort_categories_by="input", sort_by="-degree", show_percentages=True, show_counts=True, totals_plot_elements=0)

ttt.style_subsets(facecolor='blue', min_degree=1, max_degree=1)
ttt.style_subsets(facecolor='green', min_degree=2, max_degree=2)
ttt.style_subsets(facecolor='orange', min_degree=3, max_degree=3)
ttt.style_subsets(facecolor='purple', min_degree=4, max_degree=4)
ttt.style_subsets(facecolor='red', min_degree=5, max_degree=5)

ttt.plot(fig)
plt.suptitle("Heteroplasmic", size=16)
fig.savefig("plots_revisions/suppl/upset_plot_hetero.pdf", dpi=300)
plt.show()







In [ ]:
lr_specific = merged[merged['overall_status'] != 'duplex'][merged['overall_status'] != 'illumina'][merged['overall_status'] != 'duplex,illumina'][merged['overall_status'] != 'illumina,duplex']
lr_specific[['overall_status', 'heteroplasmy_category']].value_counts()

## 15 + 2 + 1 + 10 + 1 + 1 = 30/61
## 11 + 16 platform-unique

## 15 + 10 + 1 + 1 + 1 + 1 = 29 in either or both PB/ONT + dup/sr
## 4 PB/ONT shared but no others
## 11 + 9 platform-unique

In [ ]:
high_conf = lr_specific[~lr_specific['overall_status'].isin(['pacbio', 'ont'])]
high_conf['heteroplasmy_category'].value_counts()

In [ ]:
sns.set_theme(style="ticks", context="talk", font_scale=0.8)
plt.figure(figsize=(6,6))

# sns.scatterplot(
#     high_conf[high_conf['overall_status'] != 'pacbio,ont'][high_conf['heteroplasmy_category'] == 'hetero'],
#     x='AF_mito',
#     y='AF_dup'
# )


corr = high_conf[high_conf['overall_status'] != 'pacbio,ont'][high_conf['heteroplasmy_category'] == 'hetero'][['AF_mito', 'AF_dup']].corr()

common_kws = dict(scatter_kws={'s': 40, 'alpha': 0.7, 'edgecolor': 'black'},
                  line_kws={'color': 'red', 'lw': 2})

sns.regplot(data=high_conf[high_conf['overall_status'] != 'pacbio,ont'][high_conf['heteroplasmy_category'] == 'hetero'], x='AF_mito', y='AF_dup', **common_kws)
plt.xlabel('Mutect2 Heteroplasmy')
plt.ylabel('MitoScope-PB Heteroplasmy')
plt.text(0.04, 0.055, f"r = {corr.loc['AF_mito', 'AF_dup']:.3f}"),
#              transform=axes[0].transAxes,
#              ha="left", va="top")

plt.xlabel('MitoScope HF')
plt.ylabel('Duplex HF')
plt.tight_layout()
plt.savefig("plots_revisions/suppl/low_freq_vaf_concordance.pdf", dpi=300)
plt.show()

In [ ]:
#corr = final_merged_df.drop(columns=['id', 'indel', 'mean_af', 'heteroplasmy_category']).corr()
high_conf[high_conf['overall_status'] != 'pacbio,ont'][high_conf['heteroplasmy_category'] == 'hetero'][['AF_mito', 'AF_dup']].corr()

In [ ]:
# both_hetero_df = merged[merged['duplex_merge'] == 'both'][merged['heteroplasmy_category'] == 'hetero']['Donor_Tissue'].value_counts().reset_index().sort_values('Donor_Tissue')
# both_hetero_df['group'] = 'both'
# wgs_hetero_df  = merged[merged['duplex_merge'].isin(['left_only', 'both'])][merged['heteroplasmy_category'] == 'hetero']['Donor_Tissue'].value_counts().reset_index().sort_values('Donor_Tissue')
# wgs_hetero_df ['group'] = 'wgs'
# duplex_hetero_df = merged[merged['duplex_merge'].isin(['right_only', 'both'])][merged['heteroplasmy_category'] == 'hetero']['Donor_Tissue'].value_counts().reset_index().sort_values('Donor_Tissue')
# duplex_hetero_df['group'] = 'duplex'

# combined = pd.concat([both_hetero_df, wgs_hetero_df, duplex_hetero_df]).pivot(index='Donor_Tissue', columns='group', values='count').reset_index()
# combined['perc_duplex'] = combined['both'] / combined['duplex'] * 100
# combined['perc_wgs'] = combined['both'] / combined['wgs'] * 100
# combined

In [ ]:
# sns.set_theme(style="ticks", context="talk", font_scale=0.8)

# plt.figure(figsize=(6,4))
# sns.barplot(combined, x='Donor_Tissue', y='both', color='darkred')
# plt.xticks(rotation=90)
# plt.ylabel('Count')
# plt.xlabel('')
# plt.show()
# plt.figure(figsize=(6,4))
# sns.barplot(combined, x='Donor_Tissue', y='wgs', color='darkblue')
# plt.xticks(rotation=90)
# plt.ylabel('Count')
# plt.xlabel('')
# plt.show()
# plt.figure(figsize=(6,4))
# sns.barplot(combined, x='Donor_Tissue', y='duplex', color='darkgreen')
# plt.xticks(rotation=90)
# plt.ylabel('Count')
# plt.xlabel('')
# plt.show()


In [ ]:
# sns.set_theme(style="ticks", context="talk", font_scale=0.8)

# plt.figure(figsize=(6,4))
# sns.barplot(combined, x='Donor_Tissue', y='perc_duplex', color='darkgreen')
# plt.xticks(rotation=90)
# plt.ylabel('% shared duplex calls')
# plt.xlabel('')
# plt.show()

# plt.figure(figsize=(6,4))
# sns.barplot(combined, x='Donor_Tissue', y='perc_wgs', color='darkblue')
# plt.xticks(rotation=90)
# plt.ylabel('% shared MitoScope calls')
# plt.xlabel('')
# plt.show()



In [ ]:
## write HC heteroplasmic SNVs
high_conf[high_conf['heteroplasmy_category'] == 'hetero'].to_csv('tables_revisions/benchmarking_snvs_hetero_only_HC.csv')
high_conf


In [ ]:
high_conf[high_conf['Status'] == 'Cfrm [P]']

In [ ]:
high_conf[high_conf['heteroplasmy_category'] == 'hetero']['ID'].nunique()

In [ ]:
## variants not in MITOMAP
high_conf['in_MITOMAP'] = np.where(high_conf['Gene.Name'] == '', 'No', 'Yes')
high_conf[high_conf['in_MITOMAP'] == 'No']

In [ ]:
len(high_conf[high_conf['heteroplasmy_category'] == 'hetero'][high_conf['AF_mito'] < 0.05])

In [ ]:
## get df of region sizes
regions = pd.read_csv('jn_resources/GenomeLoci_MITOMAP_short.txt', sep='\t', names=['chrom', 'start', 'end', 'gene_name', 'gene_short_name', 'description'])
regions['size'] = regions['end'] - regions['start']

def classify(g):
    if g.startswith(("MT-ND", "MT-CO", "MT-ATP", "MT-CYB")):
        return "protein coding"
    elif g.startswith("MT-RNR"):
        return "rRNA"
    elif g.startswith("MT-T"):
        return "tRNA"
    elif g == "MT-CR":
        return "control region"
    else:
        return None

regions["biotype"] = regions["gene_name"].apply(classify)

region_sizes = regions.groupby('biotype')['size'].sum().reset_index()
region_sizes

In [ ]:
hypervariable_regions = list(range(16024,16365+1,1)) + list(range(73,340+1,1)) + list(range(438,574+1,1))
len(hypervariable_regions)

region_sizes.loc[len(region_sizes)] = ['hypervariable', 747]
region_sizes['size'] = np.where(region_sizes['biotype'] == 'control region', region_sizes['size'] - 747, region_sizes['size'])

region_sizes

In [ ]:
high_conf['POS'] = high_conf['ID'].str.split('-', expand=True)[1].astype(int)
high_conf['Gene.Type'] = np.where(high_conf['POS'].isin(hypervariable_regions), 'hypervariable', high_conf['Gene.Type'])
high_conf['Gene.Type'].value_counts()

In [ ]:
category_map = {
    'hypervariable': 'intergenic',
    'control region': 'intergenic',
    'noncoding': 'intergenic',
    'tRNA': 'non_coding_transcript_exon',
    'rRNA': 'non_coding_transcript_exon',
    'protein coding': 'protein coding'
}
region_sizes['for_conseq'] = region_sizes['biotype'].map(category_map)
regions_sizes_for_conseq = region_sizes.groupby('for_conseq')['size'].sum().reset_index()
regions_sizes_for_conseq

In [ ]:
## mitomap reported disease association counts
csq_counts_disease = high_conf.groupby(['Donor', 'Tissue'])['Status'].value_counts().reset_index()
csq_counts_disease['Donor_Tissue'] = csq_counts_disease['Donor'] + "_" + csq_counts_disease['Tissue']

sns.set_theme(style="ticks", context="talk", font_scale=0.9)

g = sns.catplot(
    data=csq_counts_disease[csq_counts_disease['Status'] != 'N/A'].sort_values('Status', ascending=False),
    x='Status',
    hue='Donor_Tissue',
    y='count',
    kind='bar',
    height=5,
    aspect=1.25,
    sharex=False,
    legend=True
)

sns.move_legend(g, loc='lower left', bbox_to_anchor=(0.45,0.5), title="")
sns.despine(top=False, right=False, left=False, bottom=False)
g.set_axis_labels('', 'Variant Count')
g.set_titles('{col_name}') # Set titles for each facet

# x_positions = [0, 1, 2, 3]
# custom_labels = ['Reported', 'Conflicting\nReports', 'Cfrm [P]', 'Cfrm [LP]' ]
# plt.xticks(x_positions, custom_labels)

plt.savefig(f"plots_revisions/fig4-variant_disease_associations.pdf", dpi=300)
plt.show()


In [ ]:
## gene type counts (normalized by region size)
csq_counts_type = high_conf.groupby(['Donor', 'Tissue'])['Gene.Type'].value_counts().reset_index()
csq_counts_type['Donor_Tissue'] = csq_counts_type['Donor'] + "_" + csq_counts_type['Tissue']
csq_counts_type = csq_counts_type[csq_counts_type['Gene.Type'] != 'noncoding']
csq_counts_type = pd.merge(csq_counts_type, region_sizes, left_on="Gene.Type", right_on="biotype", how="left")
csq_counts_type['per_bp_in_region'] = csq_counts_type['count'] / csq_counts_type['size']

sns.set_theme(style="ticks", context="talk", font_scale=0.8)

g = sns.catplot(
    data=csq_counts_type[csq_counts_type['Gene.Type'] != 'noncoding'],
    x='Gene.Type',
    hue='Donor_Tissue',
    y='per_bp_in_region',
    kind='bar',
    height=5,
    aspect=2,
    sharex=True,
)

g.set_axis_labels("", r'Variant (bp$^{-1}$)')
#plt.xticks(rotation=90)
plt.show()

csq_counts_type.groupby(['Gene.Type']).agg(m=('per_bp_in_region', 'mean'))

In [ ]:
## add expected + obs/exp ratios
variant_count_totals = csq_counts_type.groupby(['Donor_Tissue']).agg(total_variants=('count', 'sum')).reset_index()
csq_counts_type = pd.merge(csq_counts_type, variant_count_totals, on="Donor_Tissue", how="left")
csq_counts_type['expected_variant_freq'] = csq_counts_type['size'] / region_sizes['size'].sum()
csq_counts_type['obs_variant_freq'] = csq_counts_type['count'] / csq_counts_type['total_variants']
csq_counts_type['obs_to_exp_ratio'] = csq_counts_type['obs_variant_freq'] / csq_counts_type['expected_variant_freq']

csq_counts_type

In [ ]:
## plot enriched/depleted gene type regions (CR, protein coding, tRNA, rRNA)
sns.set_theme(style="ticks", context="talk", font_scale=0.8)

plt.figure(figsize=(6,5))

# Observed variants
sns.stripplot(
    data=csq_counts_type,
    x="Gene.Type",
    y="obs_variant_freq",
    order=['control region', 'hypervariable', 'protein coding', 'rRNA', 'tRNA'],
    hue="Tissue",
    s=8,
    legend=True,
    zorder=20  # on top
)

# Expected variants
sns.scatterplot(
    data=csq_counts_type,
    x="Gene.Type",
    y="expected_variant_freq",
    color='black',
    marker='X',
    s=80,
    label='Expected',
    zorder=30
)

sns.boxplot(
    x="Gene.Type",
    y="obs_variant_freq",
    order=['control region', 'hypervariable',  'protein coding', 'rRNA', 'tRNA'],
    data=csq_counts_type,
    color='white',
    linecolor='black',
    showfliers=False
)

# Custom x-axis labels
x_positions = [0, 1, 2, 3, 4]
custom_labels = ['Control\nRegion', 'hypervariable', 'Protein\nCoding', 'rRNA', 'tRNA']
plt.xticks(x_positions, custom_labels)

plt.ylabel("Variant Frequency")
plt.xlabel("")

plt.legend()
plt.ylim(0,1)
plt.savefig(f"plots_revisions/fig4-variant_gene_types.pdf", dpi=300)
plt.show()


In [ ]:
csq_counts_type[['Gene.Type', 'expected_variant_freq']].drop_duplicates()

In [ ]:
## chi-square results for enrichemnt analysis
from scipy.stats import chisquare

obs = csq_counts_type.groupby("Gene.Type")["count"].sum()
lengths = region_sizes.set_index("biotype")["size"]
obs = obs.loc[lengths.index]
mt_length = lengths.sum()
expected = obs.sum() * (lengths / mt_length)

chi2, pval = chisquare(f_obs=obs, f_exp=expected)
print(chi2)
print(pval)


In [ ]:
## enrichment analysis results
from scipy.stats import binomtest
from statsmodels.stats.multitest import multipletests

total_variants = obs.sum()
results = []

for region in obs.index:
    k = obs[region]
    n = total_variants
    p = lengths[region] / mt_length
    
    # test enrichment/depletion
    p_enrich = binomtest(k, n, p, alternative='greater').pvalue
    p_deplete = binomtest(k, n, p, alternative='less').pvalue
    
    expected_k = n * p
    fold_enrich = k / expected_k
    fold_dep = expected_k / k

    
    results.append({
        "region": region,
        "observed": k,
        "expected": expected_k,
        "fold_enrichment": fold_enrich,
        "fold_depletion": fold_dep,
        "p_enrich": p_enrich,
        "p_deplete": p_deplete
    })

results_df = pd.DataFrame(results)

pvals_e = results_df["p_enrich"]  # or combine enrich/deplete carefully
pvals_d = results_df["p_deplete"]  # or combine enrich/deplete carefully

results_df["p_adj_e"] = multipletests(pvals_e, method='fdr_bh')[1]
results_df["p_adj_d"] = multipletests(pvals_d, method='fdr_bh')[1]

results_df

In [ ]:
## VEP consequence plot
csq_counts_conseq = high_conf.groupby(['Donor', 'Tissue'])['Consequence'].value_counts().reset_index()
csq_counts_conseq['Donor_Tissue'] = csq_counts_conseq['Donor'] + "_" + csq_counts_conseq['Tissue']

category_map = {
    'intergenic': 'intergenic',
    'missense': 'protein coding',
    'synonymous': 'protein coding',
    'non_coding_transcript_exon': 'non_coding_transcript_exon',
}
csq_counts_conseq['mapped_category'] = csq_counts_conseq['Consequence'].map(category_map)
csq_counts_conseq = pd.merge(csq_counts_conseq, regions_sizes_for_conseq, left_on="mapped_category", right_on="for_conseq", how="left")
csq_counts_conseq['per_bp_in_region'] = csq_counts_conseq['count'] / csq_counts_conseq['size']
csq_counts_conseq


sns.set_theme(style="ticks", context="talk", font_scale=0.8)

g = sns.catplot(
    data=csq_counts_conseq,
    x='Consequence',
    hue='Donor_Tissue',
    y='per_bp_in_region',
  #  col='',
    kind='bar',
    height=5,
    aspect=2,
    sharex=False,
)

g.set_axis_labels('', r'Variant (bp$^{-1}$)')
g.set_titles('{col_name}') # Set titles for each facet
#plt.xticks(rotation=90)
#plt.yticks([0,5,10,15,20])
plt.show()

csq_counts_conseq.groupby(['Consequence']).agg(m=('per_bp_in_region', 'mean'))

In [ ]:
## strand bias plot
high_conf[['MT', 'POS', 'REF', 'ALT']] = high_conf['ID'].str.split('-', expand=True)

high_conf['nuc_change'] = high_conf['REF'] + ">" + high_conf['ALT']
high_conf['rep_strand'] = np.where(high_conf['REF'].isin(['C', 'T']), 'L-strand', 'H-strand')

conditions = [
    (high_conf['nuc_change'] == 'G>A'),
    (high_conf['nuc_change'] == 'G>T'),
    (high_conf['nuc_change'] == 'G>C'),
    (high_conf['nuc_change'] == 'A>T'),
    (high_conf['nuc_change'] == 'A>C'),
    (high_conf['nuc_change'] == 'A>G'),
]
values = ['C>T', 'C>A', 'C>G', 'T>A', 'T>G', 'T>C']

# Create a new column using np.select
high_conf['nuc_change_revised'] = np.select(conditions, values, default='-')
high_conf['nuc_change_revised'] = np.where(high_conf['nuc_change_revised'] == '-', high_conf['nuc_change'],high_conf['nuc_change_revised'])


mut_sigs = high_conf[high_conf['Gene.Type'] != 'control region'].groupby(['Donor', 'Tissue', 'rep_strand'])['nuc_change_revised'].value_counts().reset_index()
mut_sigs['Donor_Tissue'] = mut_sigs['Donor'] + "_" + mut_sigs['Tissue']

mut_sigs["prop"] = (
    mut_sigs["count"]
    / mut_sigs.groupby("Donor_Tissue")["count"].transform("sum")
)

mut_sigs = mut_sigs.sort_values('nuc_change_revised')

mut_sigs

sns.set_theme(style="ticks", context="talk", font_scale=1)

g = sns.catplot(
    data=mut_sigs,
    x="nuc_change_revised",
    order=['C>A', 'C>G', 'C>T','T>A', 'T>G', 'T>C'],
    y="prop",
    kind="bar",
    hue="rep_strand",
    row="Tissue",       
    height=2,        
    aspect=3,
    width=0.5,
    sharey=True,      
    sharex=True,
    legend=True
)

# Format axes
g.set_axis_labels('Base Substitution', '')
g.set_titles("{row_name}")
#g.set(ylim=(0, 0.5))
sns.move_legend(g, "center right", bbox_to_anchor=(0.37,0.9), title="", fontsize=16) 
plt.tight_layout()
plt.savefig(f"plots_revisions/fig4-snv_base_subs.pdf", dpi=300)
plt.show()


In [ ]:
## HF histogram plot with break in middle

# 1. Generate sample data with two distinct ranges
data_low = high_conf[high_conf['heteroplasmy_category'] == 'hetero']["AF_mito"]
data_high = high_conf[high_conf['heteroplasmy_category'] == 'homo']["AF_mito"]
data = np.concatenate([data_low, data_high])

# Define the break points
xlim1_end = 0.1
xlim2_start = 0.9

# 2. Create subplots with shared y-axis
fig, (ax, ax2) = plt.subplots(1, 2, sharey=True, figsize=(5, 6), facecolor='w', gridspec_kw={'width_ratios': [1, 1]})
fig.subplots_adjust(wspace=0.5) # adjust space between axes

# 3. Plot the histogram on both axes with different x-limits
bins = 75 # Use a consistent number of bins across both plots
ax.hist(data, bins=bins, color='darkblue', edgecolor='black')
ax2.hist(data, bins=bins, color='skyblue', edgecolor='black')

# Zoom-in / limit the view to different portions of the data
ax.set_xlim(0, xlim1_end) # Main data range
ax2.set_xlim(xlim2_start, 1) # Outlier range

# 4. Hide the spines between ax and ax2 to create the "break" appearance
ax.spines['right'].set_visible(False)
ax2.spines['left'].set_visible(False)
ax.yaxis.tick_left()
ax2.yaxis.tick_right()
ax2.tick_params(labelleft=False, right=False) # remove the left tick labels on the right plot

# 5. Add the "cut-out" diagonal lines for visual indication of the break
d = .015 # how big to make the diagonal lines in axes coordinates
kwargs = dict(transform=ax.transAxes, color='k', clip_on=False)
ax.plot((1-d, 1+d), (-d, +d), **kwargs)
ax.plot((1-d, 1+d), (1-d, 1+d), **kwargs)

kwargs.update(transform=ax2.transAxes)
ax2.plot((-d, +d), (1-d, 1+d), **kwargs)
ax2.plot((-d, +d), (-d, +d), **kwargs)

ax.xaxis.set_major_formatter(FormatStrFormatter('%.2f'))
ax2.xaxis.set_major_formatter(FormatStrFormatter('%.2f'))

ax.set_ylabel("Variant Count", fontsize=14)
fig.supxlabel("    Heteroplasmy Level", y=0.001, fontsize=14)
fig.subplots_adjust(bottom=0.12, left=0.2)
plt.savefig(f"plots_revisions/fig4-heteroplasmy_levels_with_break.pdf", dpi=300)
plt.show()


In [ ]:

conditions = [
    (high_conf['AF_mito'] > 0.95),
    (high_conf['AF_mito'] > 0.05) & (high_conf['AF_mito'] <= 0.1),
    (high_conf['AF_mito'] > 0.01) & (high_conf['AF_mito'] <= 0.05),
    (high_conf['AF_mito'] <= 0.01)
]
values = ['homo', 'het_mid', 'het_low', 'het_ultra_low']

# Create a new column using np.select
high_conf['het_group'] = np.select(conditions, values, default='Unknown')

high_conf['het_group'].value_counts()


In [ ]:
## plot stacked bar plot with homo/hetero variant counts
all_counts = high_conf.groupby(['Donor', 'Tissue', 'Donor_Tissue']).size().reset_index(name="count")
sep_counts = high_conf.groupby(['Donor', 'Tissue', 'Donor_Tissue', 'heteroplasmy_category']).size().reset_index(name="count")
homo_counts = sep_counts[sep_counts.heteroplasmy_category == 'homo']

sns.set_theme(style="ticks", context="talk", font_scale=0.8)

fig, ax = plt.subplots(layout='constrained', figsize=(5, 5))

sns.barplot(
    data=all_counts,
    x="Donor_Tissue",
    order=['ST001_Liver', 'ST001_Lung', 'ST002_Lung', 'ST002_Colon', 'ST003_Brain', 'ST004_Brain'],
    y="count",
    edgecolor='black',
    ax=ax,
    alpha=1,
    color="darkblue"
)
sns.barplot(
    data=homo_counts,
    x="Donor_Tissue",
    order=['ST001_Liver', 'ST001_Lung', 'ST002_Lung', 'ST002_Colon', 'ST003_Brain', 'ST004_Brain'],
    y="count",
    estimator=sum, 
    color="lightblue",
    alpha=1,
    edgecolor='black',
    ax=ax
)

# Get original tick labels
ticks = ax.get_xticklabels()
labels = [t.get_text() for t in ticks]
donors  = [l.split("_")[0] for l in labels]  
tissues = ['\n\n' + l.split("_")[1] for l in labels]

ax.set_xticks(range(len(labels)))
ax.set_xticklabels(donors, rotation=0)
ax.set_ylabel("Variant Count")
ax.set_xlabel("")

sec = ax.secondary_xaxis(location=0)
sec.set_xticks([0,1.5,3,4.5], labels=['\n\nLiver', '\n\nLung', '\n\nColon', '\n\nBrain'])
sec.tick_params('x', length=0)

for tick in sec.get_xticklabels():
    tick.set_fontweight("bold")
    #tick.set_fontsize(14)

midpoints = [0.5,2.5,3.5]
for x in midpoints:
    ax.axvline(x=x,color="gray",linestyle="--",linewidth=1,alpha=1,zorder=0)

# add legend
top_bar = mpatches.Patch(color='darkblue', label='Hetero')
bottom_bar = mpatches.Patch(color='skyblue', label='Homo')
plt.legend(handles=[top_bar, bottom_bar], fontsize=12)
plt.savefig(f"plots_revisions/fig4-het_vs_hom_variant_counts.pdf", dpi=300)
print(tissues)
print(donors)

In [ ]:
#high_conf.to_csv(f"tables/high_conf_snvs.csv", index=False)

In [ ]:
high_conf[high_conf['heteroplasmy_category'] == 'hetero']['Donor_Tissue'].value_counts()

In [ ]:
## distribution of HF values across samples
hetero_snvs = high_conf[high_conf['heteroplasmy_category'] == 'hetero'].sort_values(['Donor', 'Tissue'])

sns.set_theme(style="ticks", context="talk", font_scale=0.9)
plt.figure(figsize=(7,5))

sns.boxplot(hetero_snvs, 
            x='Donor_Tissue', 
            y='AF_mito', 
            hue='Donor_Tissue', 
            showmeans=True, 
            meanprops={
                "marker": "x",
                "markerfacecolor": "white", 
                "markeredgecolor": "black", 
                "markersize": "6"}, 
            showfliers=False)

plt.xticks(rotation=90)
#plt.ylim(0, 0.04)
plt.xlabel('')
plt.ylabel('Heteroplasmic Frequency')
plt.savefig(f"plots_revisions/fig4-HF_by_tissue.pdf", dpi=300,  bbox_inches='tight')
plt.show()


In [ ]:
hetero_snvs[hetero_snvs['Donor'] == 'ST001']['ID'].value_counts()

In [ ]:
hetero_snvs[hetero_snvs['Donor'] == 'ST002']

In [ ]:
## combined heatmap of SNVs with collapsed reps
sns.set_theme(style="ticks", context="talk", font_scale=0.5)

donors = ['ST001', 'ST002', 'ST003', 'ST004']
tissues = snv_df['Tissue'].unique()

heatmap_data = high_conf.pivot(index=['Donor','Tissue'], columns=['ID','POS'], values='AF_mito')
#ts = high_conf[high_conf['Donor'] == d]['Tissue'].unique()

# Sort columns by highest average AF
#heatmap_data = heatmap_data.sort_index(axis=1, level=[1,0], ascending=True)
column_order = heatmap_data.mean(axis=0).sort_values(ascending=False).index
#column_order = heatmap_data.sort_values(by=(d, ts[0]),axis=1, ascending=False).columns
heatmap_data = heatmap_data[column_order]

# Plot heatmap
plt.figure(figsize=(6, 16))
sns.heatmap(heatmap_data.T, cmap="viridis", linewidths=1,annot=False, cbar_kws={'label': 'Heteroplasmy Level'})
plt.title('')
plt.xlabel('')
plt.ylabel('')
plt.tight_layout()

#plt.savefig(f'plots/benchmark_tissues.snv_heatmap.pdf', dpi=300)
plt.show()



In [ ]:
## homo/hetero upset plots
samples = high_conf['Donor_Tissue'].unique()

new_df = high_conf.groupby(['ID', 'heteroplasmy_category'])['Donor_Tissue'].agg(','.join).reset_index()

upset_df_homo = upsetplot.from_memberships(new_df[new_df['heteroplasmy_category'] == 'homo'].Donor_Tissue.str.split(","), data=new_df[new_df['heteroplasmy_category'] == 'homo'])
upset_df_hetero = upsetplot.from_memberships(new_df[new_df['heteroplasmy_category'] == 'hetero'].Donor_Tissue.str.split(","), data=new_df[new_df['heteroplasmy_category'] == 'hetero'])

upset_df_homo['n_donors'] = (
    upset_df_homo['Donor_Tissue']
    .str.findall(r'ST\d+')
    .apply(lambda x: len(set(x)))
)
upset_df_homo = upset_df_homo.sort_values(['n_donors', 'Donor_Tissue'], ascending=[True,False])

upset_df_hetero['n_donors'] = (
    upset_df_hetero['Donor_Tissue']
    .str.findall(r'ST\d+')
    .apply(lambda x: len(set(x)))
)
upset_df_hetero['intersection_size'] = upset_df_hetero['Donor_Tissue'].str.split(',').apply(len)
upset_df_hetero = upset_df_hetero.sort_values(['intersection_size', 'n_donors', 'Donor_Tissue'], ascending=[True, True,False])

color_map = {
    1: "blue",
    2: "green",
    3: "purple",
    4: "red"
}

sns.set_theme(style="ticks", context="talk", font_scale=0.8)
fig = plt.figure(figsize=(10,6))
t = upsetplot.UpSet(upset_df_homo, element_size=None, sort_categories_by="-input", sort_by="input", show_percentages=True, totals_plot_elements=0)

for r in range(1, len(samples) + 1):
    for present_tuple in itertools.combinations(samples, r):

        present = list(present_tuple)
        absent = [s for s in samples if s not in present]

        donors = {s.split('_')[0] for s in present}
        n_donors = len(donors)

        t.style_subsets(facecolor=color_map[n_donors], present=present, absent=absent)

t.plot(fig)
plt.suptitle("Homoplasmic", size=16)
fig.savefig("plots_revisions/fig4-upset_plot_homo.pdf", dpi=300)
plt.show()


sns.set_theme(style="ticks", context="talk", font_scale=0.8)
fig = plt.figure(figsize=(10,6))
t = upsetplot.UpSet(upset_df_hetero, element_size=None, sort_categories_by="-input", sort_by="input", show_percentages=True, totals_plot_elements=0)

t.style_subsets(facecolor='blue', min_degree=1, max_degree=1)
t.style_subsets(facecolor='green', min_degree=2, max_degree=2)
t.style_subsets(facecolor='orange', min_degree=3, max_degree=3)
t.style_subsets(facecolor='purple', min_degree=4, max_degree=4)
t.style_subsets(facecolor='red', min_degree=5, max_degree=5)

t.plot(fig)
plt.suptitle("Heteroplasmic", size=16)
fig.savefig("plots_revisions/fig4-upset_plot_hetero.pdf", dpi=300)
plt.show()


In [ ]:
# ## pileup verification for discordant positions (8860,4769)
# positions = [66, 3243, 7515, 13042]
# indir="pileup_bases/benchmark"

# comb_x = pd.DataFrame()
# for s in df['Sample'].unique():
#     for p in positions:
#         x = pd.read_csv(f'{indir}/{s}.{p}.base_comp.txt', sep=" ", names=['freq', 'base'])
#         x['sample'] = s
#         x['pos'] = p
#         comb_x = pd.concat([comb_x, x])

# sr_data = pd.read_csv('../../../mutect2/smaht/illumina/benchmark/samples_full.csv', names=['sample_id', 'file'])
# for s in sr_data['sample_id'].unique():
#     for p in positions:
#         x = pd.read_csv(f'{indir}/{s}.{p}.base_comp.txt', sep=" ", names=['freq', 'base'])
#         x['sample'] = s
#         x['pos'] = p
#         comb_x = pd.concat([comb_x, x])
    
# comb_x['base'] = comb_x['base'].str.upper()
# #comb_x = comb_x[~comb_x['base'].isin(['*'])]

# comb_x_collapsed = comb_x.groupby(['sample', 'pos', 'base'])['freq'].sum().reset_index()
# comb_x_collapsed['prop'] = comb_x_collapsed['freq'] / comb_x_collapsed.groupby(['sample', 'pos'])['freq'].transform('sum')
# comb_x_collapsed[['donor', 'tissue', 'tech', 'center']] = comb_x_collapsed['sample'].str.split('-', expand=True)
# comb_x_collapsed['Donor_Tissue'] = comb_x_collapsed['donor'] + "-" + comb_x_collapsed['tissue']
# comb_x_collapsed

# idx_levels = {col: comb_x_collapsed[col].unique() for col in ['sample', 'pos', 'base']}
# new_idx = pd.MultiIndex.from_product(idx_levels.values(), names=idx_levels.keys())
# comb_x_collapsed = comb_x_collapsed.set_index(list(idx_levels)).reindex(new_idx, fill_value=0).reset_index()
# comb_x_collapsed[['donor', 'tissue', 'tech', 'center']] = comb_x_collapsed['sample'].str.split('-', expand=True)
# comb_x_collapsed['Donor_Tissue'] = comb_x_collapsed['donor'] + "-" + comb_x_collapsed['tissue']
# comb_x_collapsed


In [ ]:
# ## plot pileup results 
# sns.set_theme(style="ticks", context="talk", font_scale=0.7)

# # Use sns.catplot() instead of sns.barplot()
# g = sns.catplot(
#     data=comb_x_collapsed[(comb_x_collapsed['pos'] == 3243 ) & (comb_x_collapsed['prop'] < 0.95) & (~(comb_x_collapsed['base'].isin([',', '.', 'A'])))],
#     x='base',
#     order=['A', 'G', 'T', 'C'],
#     y='prop',
#     hue='tech',
#     col='Donor_Tissue',
#     col_wrap=6,
#     kind='strip',
#     height=2.5,
#     aspect=1,
#     dodge=True,
#     s=50
# )

# g.set_axis_labels("Base", "Proportion")
# #for ax in g.axes.flat:
#    # ax.set_yscale('log')
#    # ax.set_ylim(0,0.01)
# g.set_titles("{col_name}")
# #plt.savefig(f"plots/suppl/pileup_freqs_pathogenic_variants.3243.pdf", dpi=300)
# plt.show()


# # Use sns.catplot() instead of sns.barplot()
# g = sns.catplot(
#     data=comb_x_collapsed[(comb_x_collapsed['pos'] == 13042 ) & (comb_x_collapsed['prop'] < 0.95) & (~(comb_x_collapsed['base'].isin([',', '.', 'G'])))],
#     x='base',
#     order=['A', 'G', 'T', 'C'],
#     y='prop',
#     hue='tech',
#     col='Donor_Tissue',
#     col_wrap=6,
#     kind='strip',
#     height=2.5,
#     aspect=1,
#     dodge=True,
#     s=50
# )

# g.set_axis_labels("Base", "Proportion")
# #for ax in g.axes.flat:
#    # ax.set_yscale('log')
#    # ax.set_ylim(0,0.01)
# g.set_titles("{col_name}")
# #plt.savefig(f"plots/suppl/pileup_freqs_pathogenic_variants.13042.pdf", dpi=300)
# plt.show()

# # Use sns.catplot() instead of sns.barplot()
# g = sns.catplot(
#     data=comb_x_collapsed[(comb_x_collapsed['pos'] == 7515 ) & (comb_x_collapsed['prop'] <= 1) & (~comb_x_collapsed['base'].isin(['*', ',', '.', 'A']))],
#     x='base',
#   #  order=['A', 'G', 'T', 'C'],
#     y='prop',
#     hue='tech',
#     col='Donor_Tissue',
#     col_wrap=3,
#     kind='strip',
#     height=3.5,
#     aspect=1,
#     dodge=True,
#     s=50
# )

# g.set_axis_labels("Base", "Proportion")
# #for ax in g.axes.flat:
#    # ax.set_yscale('log')
#    # ax.set_ylim(0,0.01)
# g.set_titles("{col_name}")
# #plt.savefig(f"plots/supplementary/pileup_freqs_pathogenic_variants.13042.pdf", dpi=300)
# plt.show()




In [ ]:
# sns.set_theme(style="ticks", context="talk", font_scale=0.7)

# # Use sns.catplot() instead of sns.barplot()
# g = sns.catplot(
#     data=comb_x_collapsed[(comb_x_collapsed['pos'] == 66) &
#                           (comb_x_collapsed['tech'] != 'illumina') &
#                           (comb_x_collapsed['prop'] <= 1) &
#                           (comb_x_collapsed['Donor_Tissue'] == 'ST001-1A_LIVER') &
#                           (~comb_x_collapsed['base'].isin(['G', '.', ',', '*']))],
#     x='base',
#   #  order=['A', 'G', 'T', 'C'],
#     y='prop',
#     hue='center',
#     hue_order=['bcm', 'broad', 'nygc', 'uwsc', 'washu'],
#     col='Donor_Tissue',
#    # col_wrap=3,
#     row='tech',
#     kind='strip',
#     height=3.5,
#     aspect=1.2,
#     dodge=True,
#     s=50
# )

# g.set_axis_labels("Base", "Proportion")
# #for ax in g.axes.flat:
#    # ax.set_yscale('log')
#    # ax.set_ylim(0,0.01)
# g.set_titles("{row_name}")
# #plt.savefig(f"plots/supplementary/lowfreqhetero.ST001-Liver.66G>A.pdf", dpi=300)
# plt.show()

In [ ]:
comb_x_collapsed[(comb_x_collapsed['pos'] == 66) &
                          (comb_x_collapsed['tech'] != 'illumina') &
                          (comb_x_collapsed['prop'] <= 1) &
                          (comb_x_collapsed['Donor_Tissue'] == 'ST001-1A_LIVER')]